# LeetCode 2379 — Minimum Recolors to Get K Consecutive Black Blocks bằng Python

Link đề: https://leetcode.com/problems/minimum-recolors-to-get-k-consecutive-black-blocks/ · Độ khó: **Easy** · Chủ đề: String, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def minimumRecolors(self, blocks: str, k: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán "tìm **đoạn liên tiếp độ dài k** cần sơn lại ít ô nhất để toàn bộ thành màu đen".
2. Nhận ra bài toán thật sự là: **đếm số ký tự `'W'` trong mọi cửa sổ độ dài `k`, lấy giá trị nhỏ nhất**.
3. Dùng được lát cắt `blocks[i:i+k]`, phương thức `str.count`, mảng **prefix sum** và kỹ thuật **cửa sổ trượt** (sliding window).
4. Viết 4 cách giải: đếm thủ công từng cửa sổ, `slice.count`, prefix sum, sliding window; đối chiếu bằng test ngẫu nhiên có seed.
5. Tránh các lỗi: sai số cửa sổ (`range(n - k)` thay vì `n - k + 1`), đếm nhầm màu, lấy tổng cả chuỗi thay vì từng cửa sổ.

## 2. Đề bài & ví dụ

Cho chuỗi `blocks` gồm các ký tự `'W'` (khối trắng) và `'B'` (khối đen), và một số nguyên `k`. Mỗi **thao tác** được phép đổi một khối trắng thành khối đen.

Hãy trả về **số thao tác tối thiểu** để có **ít nhất một đoạn gồm `k` khối đen liên tiếp**.

| Ví dụ | blocks | k | Output | Giải thích |
|---|---|---|---|---|
| 1 | `"WBBWWBBWBW"` | 7 | 3 | đoạn `blocks[0..6] = "WBBWWBB"` có 3 chữ W → sơn 3 ô thì được `BBBBBBB`; không đoạn nào tốt hơn |
| 2 | `"WBWBBBW"` | 2 | 0 | đã có sẵn `"BB"` liên tiếp |

Constraints: `n == blocks.length`; `1 <= n <= 100`; `blocks[i]` là `'W'` hoặc `'B'`; `1 <= k <= n`.

Cần nhớ: đáp án luôn tồn tại vì `k <= n`; đáp án nằm trong khoảng `0..k`.

In [1]:
blocks, k = "WBBWWBBWBW", 7
print("n =", len(blocks), "k =", k)
for i in range(len(blocks) - k + 1):
    window = blocks[i:i + k]
    print(f"cửa sổ bắt đầu {i}: {window} -> số W = {window.count('W')}")

n = 10 k = 7
cửa sổ bắt đầu 0: WBBWWBB -> số W = 3
cửa sổ bắt đầu 1: BBWWBBW -> số W = 3
cửa sổ bắt đầu 2: BWWBBWB -> số W = 3
cửa sổ bắt đầu 3: WWBBWBW -> số W = 4


## 3. Kiến thức nền

### 3.1. Chuỗi là dãy ký tự có chỉ số

`s[i]` là ký tự thứ `i` (đếm từ 0). `len(s)` là độ dài. Chuỗi trong Python **không thể sửa tại chỗ**, nhưng ta chỉ cần đọc nên không sao.

In [2]:
s = "WBBW"
print(len(s), s[0], s[1], s[-1])
for i, ch in enumerate(s):
    print(i, ch)

4 W B W
0 W
1 B
2 B
3 W


### 3.2. Lát cắt (slice) `s[i:j]`

`s[i:j]` lấy các ký tự có chỉ số `i, i+1, ..., j-1` (**không gồm `j`**). Do đó `s[i:i+k]` luôn có đúng `k` ký tự (nếu không vượt quá cuối chuỗi). Lát cắt vượt biên **không báo lỗi** mà chỉ ngắn lại — hãy cẩn thận!

In [3]:
s = "WBBWWBBWBW"
print(s[0:3], s[2:5], s[7:10])
print(repr(s[8:12]), "<- vượt biên chỉ bị ngắn lại, không lỗi")

WBB BWW WBW
'BW' <- vượt biên chỉ bị ngắn lại, không lỗi


### 3.3. `str.count(sub)` đếm số lần xuất hiện

`"WBBW".count('W')` trả về `2`. Nó duyệt cả chuỗi nên tốn `O(độ dài)`.

In [4]:
print("WBBW".count('W'), "WBBW".count('B'), "BBB".count('W'))

2 2 0


### 3.4. Cửa sổ trượt: số cửa sổ độ dài `k` trong chuỗi `n` ký tự

Cửa sổ bắt đầu tại `i` phủ `i..i+k-1`. Cần `i + k - 1 <= n - 1` ⇔ `i <= n - k`. Vậy `i` chạy `0..n-k`, tức `range(n - k + 1)`: có **`n - k + 1` cửa sổ**. Quên `+1` là lỗi cực phổ biến.

In [5]:
n, k = 10, 7
print("số cửa sổ:", n - k + 1, list(range(n - k + 1)))
n, k = 5, 5
print("k = n -> đúng 1 cửa sổ:", list(range(n - k + 1)))

số cửa sổ: 4 [0, 1, 2, 3]
k = n -> đúng 1 cửa sổ: [0]


### 3.5. Prefix sum (tổng tiền tố)

Gọi `pre[i]` = số `'W'` trong `s[0:i]` (`pre[0] = 0`). Khi đó số `'W'` trong `s[i:j]` là `pre[j] - pre[i]` — **O(1)** mỗi truy vấn sau khi dựng `pre` trong O(n).

In [6]:
s = "WBBWW"
pre = [0]
for ch in s:
    pre.append(pre[-1] + (1 if ch == 'W' else 0))
print(pre)
print("số W trong s[1:4] =", pre[4] - pre[1], "| đếm trực tiếp:", s[1:4].count('W'))

[0, 1, 1, 1, 2, 3]
số W trong s[1:4] = 1 | đếm trực tiếp: 1


### 3.6. Hàm `min` và biểu thức sinh (generator)

`min(f(i) for i in range(m))` lấy giá trị nhỏ nhất của `f(i)` mà không cần tạo list. Danh sách rỗng sẽ gây lỗi — nhưng ở bài này luôn có ít nhất một cửa sổ vì `k <= n`.

In [7]:
print(min(x * x for x in [3, -2, 5]))
print(min([4, 1, 7]), min(4, 1, 7))

4
1 1


### 3.7. `True`/`False` cũng là số

Trong Python `True == 1` và `False == 0`, nên `(ch == 'W')` có thể cộng trực tiếp vào một biến đếm: `whites += (ch == 'W')`. Đây là cách viết gọn thay cho `if ch == 'W': whites += 1`. Đừng lạm dụng nếu làm code khó đọc, nhưng trong bài đếm thì rất tiện.

In [8]:
print(True + True, ('W' == 'W') + ('B' == 'W'))
whites = 0
for ch in "WBBWW":
    whites += (ch == 'W')
print("số W:", whites)

2 1
số W: 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Muốn có `k` khối đen liên tiếp, ta **chọn một đoạn dài `k`** rồi **sơn tất cả ô trắng trong đoạn đó thành đen**. Các ô đen sẵn có thì khỏi tốn công. Vậy:

> chi phí của một đoạn = số chữ `W` trong đoạn đó.

Đáp án = chi phí **nhỏ nhất** trong tất cả các đoạn dài `k`. Không cần quan tâm ngoài đoạn.

Tính tay ví dụ 1 (`"WBBWWBBWBW"`, `k = 7`), có `10 - 7 + 1 = 4` đoạn:

```
i=0: WBBWWBB -> W ở vị trí 0,3,4 -> 3
i=1: BBWWBBW -> 3 chữ W (vị trí 2,3,6) -> 3
i=2: BWWBBWB -> 3
i=3: WWBBWBW -> 4
min = 3
```

Đáp án `3` khớp với đề.

**Vì sao không cần nghĩ tới việc sơn ô ngoài đoạn?** Giả sử ta sơn thêm một ô trắng nằm ngoài đoạn đã chọn: số thao tác tăng 1 mà không giúp gì cho việc có `k` ô đen liên tiếp trong đoạn đó. Vậy phương án tối ưu không bao giờ làm thế.

**Vì sao đáp án không vượt quá `k`?** Trong trường hợp xấu nhất mọi ô trong cửa sổ đều là `W`, ta sơn đúng `k` ô. Do đó ta có thể khởi tạo `best = k` làm cận trên an toàn (Cách 1 dùng điều này).

## 5. Cách 1 — Brute Force / baseline

Với mỗi vị trí bắt đầu `i`, **tự duyệt** `k` ký tự và đếm `'W'` bằng một vòng `for` lồng. Cập nhật đáp án nhỏ nhất.

Rất trực tiếp nhưng làm lại việc nhiều: hai cửa sổ liền kề chung `k - 1` ký tự mà ta vẫn đếm lại từ đầu.

In [9]:
class SolutionBrute:
    def minimumRecolors(self, blocks: str, k: int) -> int:
        n = len(blocks)
        best = k                       # tệ nhất là sơn cả k ô
        for i in range(n - k + 1):
            whites = 0
            for j in range(i, i + k):
                if blocks[j] == 'W':
                    whites += 1
            best = min(best, whites)
        return best

print(SolutionBrute().minimumRecolors("WBBWWBBWBW", 7))
print(SolutionBrute().minimumRecolors("WBWBBBW", 2))

3
0


**Độ phức tạp:** có `n - k + 1` cửa sổ, mỗi cửa sổ đếm `k` ký tự → `O((n - k + 1) * k)`, tệ nhất cỡ `O(n^2 / 4)`. Với `n <= 100` thì rất nhanh, nhưng nút thắt là **đếm lại phần chung giữa hai cửa sổ liền nhau**. Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

Hai cửa sổ liền nhau `[i, i+k-1]` và `[i+1, i+k]` chỉ khác nhau **hai ô**: ô `i` rời đi, ô `i+k` bước vào. Do đó:

- `whites(i+1) = whites(i) - (blocks[i] == 'W') + (blocks[i+k] == 'W')`.

**Quy tắc "nếu… thì…":**

- Nếu ô rời đi là `'W'` → số W giảm 1.
- Nếu ô mới vào là `'W'` → số W tăng 1.
- Cả hai cùng là `'W'` hoặc cùng là `'B'` → không đổi.

Invariant: sau mỗi bước, biến `whites` luôn bằng đúng số `'W'` trong cửa sổ hiện tại. Ngoài ra còn cách thứ hai cùng tinh thần: **prefix sum** cho `whites(i) = pre[i+k] - pre[i]` trong O(1).

In [10]:
blocks, k = "WBBWWBBWBW", 7
whites = blocks[:k].count('W')
print("cửa sổ 0:", blocks[:k], whites)
for i in range(1, len(blocks) - k + 1):
    out, inn = blocks[i - 1], blocks[i + k - 1]
    whites += (inn == 'W') - (out == 'W')
    print(f"i={i}: ra '{out}', vào '{inn}' -> {whites} | kiểm tra: {blocks[i:i+k].count('W')}")

cửa sổ 0: WBBWWBB 3
i=1: ra 'W', vào 'W' -> 3 | kiểm tra: 3
i=2: ra 'B', vào 'B' -> 3 | kiểm tra: 3
i=3: ra 'B', vào 'W' -> 4 | kiểm tra: 4


## 7. Các cách tiếp theo

Ta có 4 cách tổng cộng: (1) Brute đếm tay, (2) `slice.count`, (3) prefix sum, (4) sliding window.

### Cách 2 — Lát cắt + `count`

Giữ nguyên ý "duyệt mọi cửa sổ" nhưng để Python đếm: `blocks[i:i+k].count('W')`. Code ngắn gọn, đếm bằng C nên nhanh trong thực tế dù vẫn `O(n*k)`.

In [11]:
class SolutionSlice:
    def minimumRecolors(self, blocks: str, k: int) -> int:
        return min(blocks[i:i + k].count('W') for i in range(len(blocks) - k + 1))

print(SolutionSlice().minimumRecolors("WBBWWBBWBW", 7))

3


**Code nộp (Cách 2)** — chỉ để thấy dạng chuẩn; Cách 4 mới là bản cuối ở mục 9.

In [12]:
from typing import List

class Solution:
    def minimumRecolors(self, blocks: str, k: int) -> int:
        return min(blocks[i:i + k].count('W') for i in range(len(blocks) - k + 1))

print(Solution().minimumRecolors("WBWBBBW", 2))

0


Độ phức tạp Cách 2: Time `O((n - k + 1) * k)`, Space `O(k)` cho mỗi lát cắt tạm (không giữ lại) → xem là `O(k)`.

### Cách 3 — Prefix sum

Bước 1: dựng `pre` với `pre[i+1] = pre[i] + (blocks[i] == 'W')`. Bước 2: với mỗi `i`, số W của cửa sổ là `pre[i+k] - pre[i]`. Lấy min.

In [13]:
class SolutionPrefix:
    def minimumRecolors(self, blocks: str, k: int) -> int:
        n = len(blocks)
        pre = [0] * (n + 1)
        for i, ch in enumerate(blocks):
            pre[i + 1] = pre[i] + (ch == 'W')
        return min(pre[i + k] - pre[i] for i in range(n - k + 1))

print(SolutionPrefix().minimumRecolors("WBBWWBBWBW", 7))
print(SolutionPrefix().minimumRecolors("WBWBBBW", 2))

3
0


Độ phức tạp Cách 3: Time `O(n)`, Space `O(n)` cho mảng `pre`. Ưu điểm: dễ mở rộng cho nhiều truy vấn độ dài khác nhau.

### Cách 4 — Sliding window (cách chuẩn)

Đếm W của cửa sổ đầu, rồi trượt từng bước, chỉ cộng/trừ hai ô thay đổi. Chỉ giữ một biến số → `O(1)` bộ nhớ.

In [14]:
class SolutionWindow:
    def minimumRecolors(self, blocks: str, k: int) -> int:
        whites = blocks[:k].count('W')
        best = whites
        for i in range(k, len(blocks)):          # i là ô mới vào
            if blocks[i] == 'W':
                whites += 1
            if blocks[i - k] == 'W':             # ô rời đi
                whites -= 1
            best = min(best, whites)
        return best

print(SolutionWindow().minimumRecolors("WBBWWBBWBW", 7))
print(SolutionWindow().minimumRecolors("WBWBBBW", 2))

3
0


Độ phức tạp Cách 4: Time `O(n)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 4 với `blocks = "WBBWWBBWBW"`, `k = 7`.

| i (vào) | ô vào | ô ra (`i-k`) | whites | best |
|---|---|---|---|---|
| khởi tạo | – | – | 3 | 3 |
| 7 | `W` | `blocks[0]=W` | 3 | 3 |
| 8 | `B` | `blocks[1]=B` | 3 | 3 |
| 9 | `W` | `blocks[2]=B` | 4 | 3 |

In [15]:
def trace_window(blocks, k):
    whites = blocks[:k].count('W')
    best = whites
    print(f"khởi tạo: cửa sổ={blocks[:k]} whites={whites} best={best}")
    for i in range(k, len(blocks)):
        vao, ra = blocks[i], blocks[i - k]
        if vao == 'W':
            whites += 1
        if ra == 'W':
            whites -= 1
        best = min(best, whites)
        print(f"i={i}: vào={vao} ra={ra} cửa sổ={blocks[i-k+1:i+1]} whites={whites} best={best}")
    return best

print("đáp án:", trace_window("WBBWWBBWBW", 7))

khởi tạo: cửa sổ=WBBWWBB whites=3 best=3
i=7: vào=W ra=W cửa sổ=BBWWBBW whites=3 best=3
i=8: vào=B ra=B cửa sổ=BWWBBWB whites=3 best=3
i=9: vào=W ra=B cửa sổ=WWBBWBW whites=4 best=3
đáp án: 3


## 9. Code nộp LeetCode

Bản cuối dùng sliding window (`O(n)` thời gian, `O(1)` bộ nhớ).

In [16]:
from typing import List

class Solution:
    def minimumRecolors(self, blocks: str, k: int) -> int:
        whites = blocks[:k].count('W')
        best = whites
        for i in range(k, len(blocks)):
            if blocks[i] == 'W':
                whites += 1
            if blocks[i - k] == 'W':
                whites -= 1
            if whites < best:
                best = whites
        return best

print(Solution().minimumRecolors("WBBWWBBWBW", 7), Solution().minimumRecolors("WBWBBBW", 2))

3 0


## 10. Vì sao thuật toán đúng

- **Đủ:** một cách làm hợp lệ luôn chứa một đoạn `k` ô đen liên tiếp. Với đoạn đó, các ô trắng nằm trong đoạn đều **phải** được sơn; sơn thêm ô ngoài đoạn là thừa. Nên số thao tác tối thiểu cho đoạn cố định bằng đúng số `'W'` trong đoạn.
- **Tối ưu:** thử mọi đoạn rồi lấy min nên không bỏ sót phương án.
- **Sliding window:** theo invariant ở mục 6, `whites` luôn bằng số W của cửa sổ hiện tại. Cửa sổ ban đầu được khởi tạo đúng bằng `count`, và mỗi lần trượt đúng thay đổi 2 ô.
- **Không có phương án rẻ hơn:** nếu tồn tại cách sơn `t` ô để tạo ra `k` ô đen liên tiếp, thì đoạn `k` ô đó ban đầu có tối đa `t` ô trắng (các ô trắng còn lại trong đoạn đều phải được sơn). Do đó `min` trên mọi đoạn `<= t`, tức không thể tệ hơn phương án bất kỳ.
- **Biên:** `k == n` có đúng 1 cửa sổ, vòng `range(k, n)` rỗng và ta trả về `whites` ban đầu.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(blocks)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute đếm tay | `O((n-k+1)*k)` | `O(1)` |
| 2. `slice.count` | `O((n-k+1)*k)` | `O(k)` |
| 3. Prefix sum | `O(n)` | `O(n)` |
| 4. Sliding window | `O(n)` | `O(1)` |

Với `n <= 100` mọi cách đều nhanh; sliding window là cách "đáng học" vì dùng lại cho các bài chuỗi/mảng lớn hơn. Không có khác biệt average/worst: thuật toán luôn duyệt đủ.

## 12. Bộ test

Test 1: ví dụ của đề và edge case.

In [17]:
tests = [
    ("WBBWWBBWBW", 7, 3),
    ("WBWBBBW", 2, 0),
    ("W", 1, 1),          # n = 1, phải sơn
    ("B", 1, 0),          # n = 1, đã đen
    ("WWWWW", 5, 5),      # k = n, toàn trắng
    ("BBBBB", 3, 0),      # toàn đen
    ("WWWBB", 2, 0),      # cửa sổ cuối là đáp án (bắt lỗi range(n-k))
    ("BBWWW", 3, 1),
    ("WBWBW", 3, 1),
]
impls = [SolutionBrute(), SolutionSlice(), SolutionPrefix(), SolutionWindow(), Solution()]
for b, k, exp in tests:
    for im in impls:
        assert im.minimumRecolors(b, k) == exp, (type(im).__name__, b, k, exp)
print("tất cả test cố định OK:", len(tests), "test x", len(impls), "cách")

tất cả test cố định OK: 9 test x 5 cách


Test 2: ngẫu nhiên có seed, đối chiếu cả 5 cài đặt (tôn trọng constraint: `1 <= n <= 100`, `1 <= k <= n`).

In [18]:
import random
random.seed(2379)
for _ in range(3000):
    n = random.randint(1, 100)
    k = random.randint(1, n)
    blocks = ''.join(random.choice('WB') for _ in range(n))
    res = {im.minimumRecolors(blocks, k) for im in impls}
    assert len(res) == 1, (blocks, k, res)
    assert 0 <= next(iter(res)) <= k
print("3000 test ngẫu nhiên OK, mọi cách cho cùng đáp án")

3000 test ngẫu nhiên OK, mọi cách cho cùng đáp án

Test 3: phản ví dụ — thêm vài trường hợp "bẫy" (chuỗi xen kẽ, đoạn đen ở rìa trái/phải).

In [19]:
traps = [("BWBWBWBW", 2, 1), ("BWBWBWBW", 1, 0), ("WWWWBBBB", 4, 0), ("BBBBWWWW", 4, 0), ("WBBBW", 5, 2)]
for b, k, exp in traps:
    assert Solution().minimumRecolors(b, k) == exp, (b, k)
print("phản ví dụ OK")

phản ví dụ OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `range(n - k)` → bỏ sót cửa sổ cuối

In [20]:
def wrong_off_by_one(blocks, k):
    return min(blocks[i:i + k].count('W') for i in range(len(blocks) - k))   # SAI: thiếu +1

b, k = "WWWBB", 2
print("sai :", wrong_off_by_one(b, k))
print("đúng:", Solution().minimumRecolors(b, k))
assert wrong_off_by_one(b, k) != Solution().minimumRecolors(b, k)

sai : 1
đúng: 0


Vì sao sai: cửa sổ cuối (`"BB"`) không bao giờ được xét, nên đáp án 1 thay vì 0. Và nếu `k == n` thì `range(0)` rỗng → `min()` báo lỗi. **Sửa:** `range(n - k + 1)`.

### Lỗi 2: đếm cả chuỗi thay vì từng cửa sổ

In [21]:
def wrong_global(blocks, k):
    return min(blocks.count('W'), k)    # SAI: đếm W của toàn chuỗi

b, k = "WBBWWBBWBW", 7
print("sai :", wrong_global(b, k))
print("đúng:", Solution().minimumRecolors(b, k))
assert wrong_global(b, k) != Solution().minimumRecolors(b, k)

sai : 5
đúng: 3


Vì sao sai: các ô trắng nằm ngoài đoạn được chọn không cần sơn. **Sửa:** chỉ đếm W **bên trong** từng cửa sổ.

### Lỗi 3: quên trừ ô rời đi khi trượt

In [22]:
def wrong_no_remove(blocks, k):
    whites = blocks[:k].count('W')
    best = whites
    for i in range(k, len(blocks)):
        if blocks[i] == 'W':
            whites += 1               # SAI: không trừ ô rời đi
        best = min(best, whites)
    return best

b, k = "WBBWWBBWBW", 7
print("sai :", wrong_no_remove(b, k))
print("đúng:", Solution().minimumRecolors(b, k))
b2, k2 = "WWBBB", 3
print("ví dụ thứ hai sai/đúng:", wrong_no_remove(b2, k2), Solution().minimumRecolors(b2, k2))
assert wrong_no_remove(b2, k2) != Solution().minimumRecolors(b2, k2)

sai : 3
đúng: 3
ví dụ thứ hai sai/đúng: 2 0


Vì sao sai: `whites` chỉ tăng, không phản ánh cửa sổ hiện tại nữa → vượt giá trị thật. **Sửa:** mỗi lần vào một ô thì phải cho ô cũ rời đi (`blocks[i-k]`).

### Lỗi 4 (bonus): đếm nhầm màu

Nếu đếm `'B'` rồi lấy `min` thay vì `max`, bạn sẽ tìm cửa sổ **nhiều W nhất**. Hãy nhớ: chi phí = số `'W'`, cần **min**; hoặc đếm `'B'` và dùng `k - max`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Chữ khóa **"đoạn liên tiếp độ dài k"** → nghĩ ngay tới **sliding window** hoặc prefix sum.
- Đề cho một đại lượng đếm được trong cửa sổ (ở đây: số W) → duy trì nó bằng cộng/trừ hai đầu.
- Với `n <= 100` bạn có thể nộp cách `slice.count` ngắn gọn mà vẫn qua; nhưng hãy tập viết sliding window để quen tay.
- Luôn kiểm tra `n - k + 1` cửa sổ và trường hợp `k == n`.
- Dùng assert so sánh với brute force trước khi nộp.
- Khi gặp bài "đoạn con liên tiếp" mà đề cho độ dài **cố định**, dùng cửa sổ trượt độ dài cố định (đơn giản nhất). Nếu độ dài **thay đổi** (tìm đoạn dài nhất/ngắn nhất thoả điều kiện) thì chuyển sang hai con trỏ `left/right`.
- Cẩn thận lỗi "lệch một" (off-by-one): hãy tự đếm bằng ví dụ nhỏ như `n = 3, k = 2` (đúng 2 cửa sổ) trước khi tin vào biểu thức `n - k + 1`.
- Khi `n <= 100`, ưu tiên code **dễ đúng** hơn code ngắn nhất; nhưng cũng đừng bỏ qua bài học về sliding window vì nó là nền cho nhiều bài Medium.

## 15. Biến thể của bài

1. Đổi cho phép: mỗi thao tác đổi **một khối bất kỳ** (cả B thành W) — nhưng muốn có k khối **cùng màu** (bất kỳ màu nào): lấy `min(min W, min B)` trên các cửa sổ.
2. Trả về **vị trí bắt đầu** của một đoạn tối ưu (thay vì số thao tác).
3. Nhiều truy vấn `k` khác nhau trên cùng một chuỗi → prefix sum thắng.
4. Mảng nhị phân, tìm đoạn dài nhất có tối đa `m` số 0 (bài "Max Consecutive Ones III").
5. Cửa sổ vòng tròn: chuỗi nối đuôi nhau thành vòng (nhân đôi chuỗi).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute đếm tay | đếm lại từng cửa sổ bằng vòng lặp | `O((n-k+1)k)` | `O(1)` | học khái niệm, `n` rất nhỏ |
| 2. `slice.count` | lát cắt + `count` | `O((n-k+1)k)` | `O(k)` | code ngắn, `n` nhỏ như bài này |
| 3. Prefix sum | `pre[i+k]-pre[i]` | `O(n)` | `O(n)` | nhiều truy vấn / nhiều `k` |
| 4. Sliding window | cộng ô vào, trừ ô ra | `O(n)` | `O(1)` | chuẩn, tiết kiệm bộ nhớ |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `"BWBW"`, `k = 2` trả về bao nhiêu?

**Bài 2 (tính tay):** với `blocks = "WWBWBBW"`, `k = 3`, liệt kê số W của từng cửa sổ và đáp án.

**Bài 3 (điền TODO):** hoàn thành hàm `count_w_prefix` dựng prefix số `'W'`.

**Bài 4 (tự viết lại):** viết hàm `best_start(blocks, k)` trả về **chỉ số bắt đầu nhỏ nhất** của cửa sổ tốn ít thao tác nhất.

**Bài 5 (mini challenge):** viết `min_recolors_any(blocks, k)` — được phép tạo `k` khối liên tiếp **cùng màu bất kỳ** (cả đen lẫn trắng).

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().minimumRecolors("BWBW", 2))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_w_prefix(blocks):
    pre = [0]
    for ch in blocks:
        # TODO: thêm vào pre số W tính đến hiện tại
        pass
    return pre

1


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** mỗi bài chỉ cần quy về "đếm W trong từng đoạn dài k". Nhớ có `n - k + 1` đoạn.

**Mức 2 (vừa):** Bài 3: `pre.append(pre[-1] + (ch == 'W'))`. Bài 4: dùng `min` trên `range(n-k+1)` với `key`, hoặc duyệt và chỉ cập nhật khi **nhỏ hơn hẳn** để giữ chỉ số đầu tiên. Bài 5: chạy hai lần, một lần đếm W, một lần đếm B.

**Mức 3 (gần đáp án):** Bài 4: `min(range(n-k+1), key=lambda i: blocks[i:i+k].count('W'))` — `min` trả về phần tử đầu tiên khi bằng nhau. Bài 5: kết quả là `min(min_w, min_b)` với `min_b` là số `'B'` nhỏ nhất ở cửa sổ.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1
assert Solution().minimumRecolors("BWBW", 2) == 1

# Bài 2: WWBWBBW, k=3 -> các cửa sổ WWB, WBW, BWB, WBB, BBW  (số W: 2, 2, 1, 1, 1)
b = "WWBWBBW"
counts = [b[i:i + 3].count('W') for i in range(len(b) - 3 + 1)]
print("số W từng cửa sổ:", counts, "-> đáp án", min(counts))
assert counts == [2, 2, 1, 1, 1] and min(counts) == 1

# Bài 3
def count_w_prefix(blocks):
    pre = [0]
    for ch in blocks:
        pre.append(pre[-1] + (ch == 'W'))
    return pre
assert count_w_prefix("WBBWW") == [0, 1, 1, 1, 2, 3]

# Bài 4
def best_start(blocks, k):
    return min(range(len(blocks) - k + 1), key=lambda i: blocks[i:i + k].count('W'))
assert best_start("WWBWBBW", 3) == 2
assert best_start("WBWBBBW", 2) == 3

# Bài 5
def min_recolors_any(blocks, k):
    best = k
    for i in range(len(blocks) - k + 1):
        w = blocks[i:i + k].count('W')
        best = min(best, w, k - w)
    return best
assert min_recolors_any("WBWBW", 3) == 1
assert min_recolors_any("WWBBWW", 6) == 2
assert min_recolors_any("BBBB", 4) == 0
print("đáp án bài tập OK")

số W từng cửa sổ: [2, 2, 1, 1, 1] -> đáp án 1
đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template sliding window đếm trong cửa sổ cố định `k`:**

```python
cnt = count(arr[:k])          # khởi tạo cửa sổ đầu
best = cnt
for i in range(k, len(arr)):
    cnt += vao(arr[i])         # ô mới vào
    cnt -= ra(arr[i - k])      # ô cũ rời đi
    best = min(best, cnt)      # hoặc max tuỳ bài
return best
```

**Checklist trước khi submit:**

- [ ] Có đúng `n - k + 1` cửa sổ (kể cả cửa sổ cuối)?
- [ ] Trường hợp `k == n` và `n == 1` chạy đúng?
- [ ] Đang tối thiểu hoá số **W** (không phải số B)?
- [ ] Mỗi lần trượt có cộng ô vào **và** trừ ô ra?
- [ ] Đã so sánh với brute force bằng test ngẫu nhiên có seed?
- [ ] Chữ ký đúng `minimumRecolors(self, blocks: str, k: int) -> int`?